# Lab 7 — Residual audio quantization and a TTS evaluation harness

COMP-672 · John Glossner

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glossner/COMP-672-Labs/blob/main/labs/week07_starter.ipynb)

**First: File → Save a copy in Drive.** Sign in to Google to run and save. Rename your copy, for example `COMP672_Lab07_YourName.ipynb`, and work in that copy. 


## Install the lab environment

Use Colab’s **default Python CPU runtime**; you do not need to select an older Python runtime. 


In [ ]:
import sys
import subprocess
import importlib.metadata as metadata

if not (3, 11) <= sys.version_info[:2] <= (3, 14):
    raise RuntimeError("This lab supports Python 3.11–3.14. Reopen the latest notebook from the course README if your runtime is newer.")

print("Runtime Python:", sys.version.split()[0])

LAB_PACKAGES = {'numpy': '2.1.3', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'matplotlib': '3.10.0'}
# Python 3.14 needs newer binary wheels; retain the established pins on 3.11–3.13.
if sys.version_info[:2] >= (3, 14):
    LAB_PACKAGES.update({'numpy': '2.3.5', 'scikit-learn': '1.7.2', 'matplotlib': '3.10.8'})
def installed_version(name):
    try:
        return metadata.version(name).split("+")[0]
    except metadata.PackageNotFoundError:
        return None

needed = [f"{name}=={version}" for name, version in LAB_PACKAGES.items()
          if name != "torch" and installed_version(name) != version]
if needed:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--only-binary=:all:", *needed])
print("Package setup complete. Restart the session if Colab requests it.")


In [ ]:
import platform
import importlib

LAB_VERSIONS = {"python": platform.python_version()}
for distribution, module_name in [("numpy", "numpy"), ("scipy", "scipy"),
                                  ("scikit-learn", "sklearn"), ("matplotlib", "matplotlib")]:
    module = importlib.import_module(module_name)
    LAB_VERSIONS[distribution] = module.__version__
for name, expected in LAB_PACKAGES.items():
    if LAB_VERSIONS[name].split("+")[0] != expected:
        raise RuntimeError(f"Restart the session: {name} is already imported at a different version.")
print(LAB_VERSIONS)
print("CPU environment ready. Synthetic data will be generated by the lab functions.")


# Lab 7 — Residual audio quantization and a TTS evaluation harness

Chapters 17

## Implementation tasks

1. Implement nearest-neighbor residual quantization; return indices and reconstructed vectors, and check residuals after each stage.

2. Implement fixed-length nominal rate, using ceiling(log2 K) for non-power-of-two codebooks and stating excluded overhead.

3. Implement a paired bootstrap over sentences for simulated ratings. Keep rater identities fixed and state the resulting inferential limitation.


Functions to complete: `residual_quantize`, `nominal_bitrate`, `paired_sentence_bootstrap`.

## Required experiments

- Compare one and two quantization stages for reconstruction error and nominal rate. Plot or tabulate the tradeoff and show at least three reconstructions.

- Analyze the provided simulated paired ratings, reporting means and a sentence-bootstrap interval. Prepare a blank, blinded listening-study protocol for intelligibility, naturalness, and speaker similarity; do not report synthetic values as human judgments.

## Brief analysis in your notebook or code comments

1. Which parts of a neural codec are absent from the scalar laboratory analogue?

2. Why do codec MSE and automatic transcription error not fully measure TTS quality?

3. What changes to the resampling design would allow generalization beyond the particular raters?


Answer each question in approximately 2–4 sentences beside the results. This is part of the lab, with no separate report.


## Starter code

Complete the exercise functions below. Keep the supplied checks and add the cases required by the handout.

In [ ]:
"""Week 7 student starter. Tiny residual quantizer and simulated rating analysis."""


In [ ]:
import math


In [ ]:
import numpy as np


In [ ]:
def residual_quantize(x,codebooks):
    """x N,d; each codebook K,d. Return N,Q indices and N,d reconstruction."""
    raise NotImplementedError("Complete this laboratory task.")


In [ ]:
def nominal_bitrate(frames_per_second,entries_per_codebook):
    """Fixed-length code indices, no entropy coding or transport overhead."""
    raise NotImplementedError("Complete this laboratory task.")


In [ ]:
def paired_sentence_bootstrap(a,b,iterations=1000,seed=17):
    """a,b shape sentence,rater. Resample sentences, keeping raters fixed.
    CI covers sentence sampling only; not generalization to a population of raters.
    """
    raise NotImplementedError("Complete this laboratory task.")


In [ ]:
def checks():
    books=[np.array([[0.],[.5],[1.]]),np.array([[-.25],[0.],[.25]])]
    ix,reconstruction=residual_quantize(np.array([[.7]]),books)
    assert ix.tolist()==[[1,2]] and np.isclose(reconstruction[0,0],.75)
    assert nominal_bitrate(50,[1024]*8)==4000
    # Includes non-power-of-two sizes so the fixed-length coding convention is explicit.
    assert nominal_bitrate(10,[3])==20
    x=np.arange(12).reshape(4,3)
    result=paired_sentence_bootstrap(x,x)
    assert result['mean_difference']==0 and result['ci95']==[0.,0.]


In [ ]:
def experiment():
    rng=np.random.default_rng(17); x=rng.uniform(-1,1,(200,1))
    books=[np.linspace(-1,1,8)[:,None],np.linspace(-.15,.15,8)[:,None]]
    one=residual_quantize(x,books[:1])[1]; two=residual_quantize(x,books)[1]
    # Simulated ratings exercise the analysis pipeline. These are not human listening results.
    a=np.clip(rng.normal(3.8,.5,(24,4)),1,5); b=np.clip(a-rng.normal(.2,.3,(24,4)),1,5)
    return {'one_stage_mse':float(np.mean((x-one)**2)),'two_stage_mse':float(np.mean((x-two)**2)),'one_stage_bits_per_second':nominal_bitrate(50,[8]),'two_stage_bits_per_second':nominal_bitrate(50,[8,8]),'simulated_sentence_bootstrap':paired_sentence_bootstrap(a,b),'note':'Scalar quantization is a teaching analogue of residual VQ, not a trained neural speech codec or a TTS synthesizer.'}


## Run checks and the supplied experiment

Complete the exercise functions first. A `NotImplementedError` here is expected in an untouched starter. The result dictionary covers the supplied baseline; the handout also requires comparisons and analysis that you must add below.


In [ ]:
import time

checks()
_started = time.perf_counter()
results = experiment()
baseline_runtime_seconds = time.perf_counter() - _started
results


## Your required comparisons

Add the additional experiments, checks, plots/tables, measured runtimes, and clearly labeled results required by the handout. Add their JSON-compatible summaries to `additional_results`.


In [ ]:
# TODO: implement and run the remaining required comparisons here.
additional_results = {}


## Brief analysis

1. Which parts of a neural codec are absent from the scalar laboratory analogue?

2. Why do codec MSE and automatic transcription error not fully measure TTS quality?

3. What changes to the resampling design would allow generalization beyond the particular raters?

Write approximately 2–4 sentences for each question in the next cell, tied to your measured results.

1. _Your response:_

2. _Your response:_

3. _Your response:_


## Save and submit

From Colab: **Share → General access → Anyone with the link → Viewer**. Click **Copy link**. Paste the link into the Canvas assignment submission.
